In [1]:
import pandas as pd
import numpy as np
import requests

def fetch_and_preprocess_test_data():
    print("⏳ Đang cào dữ liệu thực tế 7 ngày qua từ Open-Meteo...")
    lat, lon = 10.8231, 106.6297
    params = f"latitude={lat}&longitude={lon}&timezone=Asia%2FBangkok&past_days=7&forecast_days=0"
    
    url_weather = f"https://api.open-meteo.com/v1/forecast?{params}&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m"
    url_air = f"https://air-quality-api.open-meteo.com/v1/air-quality?{params}&hourly=pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone,sulphur_dioxide"
    
    res_w = requests.get(url_weather).json()['hourly']
    res_a = requests.get(url_air).json()['hourly']

    df = pd.DataFrame({
        'time': pd.to_datetime(res_w['time']),
        'PM2.5': res_a['pm2_5'],
        'PM10': res_a['pm10'],
        'NO2': res_a['nitrogen_dioxide'],
        'CO': res_a['carbon_monoxide'],
        'SO2': res_a['sulphur_dioxide'],
        'O3': res_a['ozone'],
        'Temperature': res_w['temperature_2m'],
        'Humidity': res_w['relative_humidity_2m'],
        'Rain': res_w['precipitation'],
        'Wind_Speed': res_w['wind_speed_10m'],
        'Wind_Dir': res_w['wind_direction_10m']
    })
    
    df.set_index('time', inplace=True)
    df.interpolate(method='linear', inplace=True)

    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365)
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

    wd_rad = df['Wind_Dir'] * np.pi / 180
    df['Wind_sin'] = np.sin(wd_rad)
    df['Wind_cos'] = np.cos(wd_rad)

    # TẠO LAG 24 TIẾNG CHO PM2.5
    lag_dict = {f'PM2.5_lag_{lag}': df['PM2.5'].shift(lag) for lag in range(1, 25)}
    df = pd.concat([df, pd.DataFrame(lag_dict, index=df.index)], axis=1)

    # Xóa các dòng NaN do shift tạo ra
    df.dropna(inplace=True) 
    
    return df

# Chạy hàm và lưu vào biến df_test
df_test = fetch_and_preprocess_test_data()

# ==========================================
# SOI DATA TẠI ĐÂY NÈ BRO
# ==========================================
print(f"✅ Data Shape (Dòng, Cột): {df_test.shape}")
print("\n📌 THỨ TỰ CỘT HIỆN TẠI (Check kỹ xem có khớp 100% với lúc train không!):")
for i, col in enumerate(df_test.columns):
    print(f"{i}: {col}")

print("\n🔍 XEM THỬ 3 DÒNG ĐẦU:")
display(df_test.head(3)) # Dùng display nếu chạy trên Jupyter, nếu IDE thường thì print(df_test.head(3))

⏳ Đang cào dữ liệu thực tế 7 ngày qua từ Open-Meteo...
✅ Data Shape (Dòng, Cột): (144, 42)

📌 THỨ TỰ CỘT HIỆN TẠI (Check kỹ xem có khớp 100% với lúc train không!):
0: PM2.5
1: PM10
2: NO2
3: CO
4: SO2
5: O3
6: Temperature
7: Humidity
8: Rain
9: Wind_Speed
10: Wind_Dir
11: hour_sin
12: hour_cos
13: doy_sin
14: doy_cos
15: is_weekend
16: Wind_sin
17: Wind_cos
18: PM2.5_lag_1
19: PM2.5_lag_2
20: PM2.5_lag_3
21: PM2.5_lag_4
22: PM2.5_lag_5
23: PM2.5_lag_6
24: PM2.5_lag_7
25: PM2.5_lag_8
26: PM2.5_lag_9
27: PM2.5_lag_10
28: PM2.5_lag_11
29: PM2.5_lag_12
30: PM2.5_lag_13
31: PM2.5_lag_14
32: PM2.5_lag_15
33: PM2.5_lag_16
34: PM2.5_lag_17
35: PM2.5_lag_18
36: PM2.5_lag_19
37: PM2.5_lag_20
38: PM2.5_lag_21
39: PM2.5_lag_22
40: PM2.5_lag_23
41: PM2.5_lag_24

🔍 XEM THỬ 3 DÒNG ĐẦU:


,PM2.5,PM10,NO2,CO,SO2,O3,Temperature,Humidity,Rain,Wind_Speed,...,PM2.5_lag_15,PM2.5_lag_16,PM2.5_lag_17,PM2.5_lag_18,PM2.5_lag_19,PM2.5_lag_20,PM2.5_lag_21,PM2.5_lag_22,PM2.5_lag_23,PM2.5_lag_24
time,,,,,,,,,,,,,,,,,,,,,
2026-03-16 00:00:00,15.7,16.8,24.4,444.0,7.4,58.0,24.2,89,0.0,7.6,...,25.3,28.1,41.3,47.1,42.7,40.2,36.9,33.6,31.9,30.5
2026-03-16 01:00:00,15.6,16.7,24.4,378.0,7.0,56.0,24.1,90,0.0,5.7,...,23.8,25.3,28.1,41.3,47.1,42.7,40.2,36.9,33.6,31.9
2026-03-16 02:00:00,16.6,17.8,25.2,328.0,7.2,53.0,24.0,91,0.0,3.9,...,20.7,23.8,25.3,28.1,41.3,47.1,42.7,40.2,36.9,33.6


In [3]:
import joblib
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ==========================================
# NẠP MODEL VÀ SCALER
# ==========================================
print("🔄 Đang nạp Model và Scaler...")
try:
    model_xgb = joblib.load('xgboost_model.pkl')
    model_ridge = joblib.load('ridge_model.pkl')
    
    # Ở đây tui xài 1 scaler chung theo hình bro gửi. 
    # Nếu lúc Train bro xài scaler_X và scaler_y riêng thì nhớ load cho đúng nha!
    scaler = joblib.load('scaler.pkl') 
    
except Exception as e:
    print(f"❌ Lỗi nạp file: {e}")

# ==========================================
# CHUẨN BỊ X VÀ y TỪ df_test (TỪ CELL 1)
# ==========================================
# Lấy toàn bộ các cột làm features (X)
X_test_raw = df_test.values 

# Target (y) chính là nồng độ PM2.5 thực tế ở chính giờ đó
y_test_real = df_test['PM2.5'].values

# Scale X (quan trọng: dùng transform, KHÔNG dùng fit_transform)
X_test_scaled = scaler.transform(X_test_raw)

print("\n" + "="*50)
print("📊 BẢNG ĐIỂM NGHIỆM THU THỰC TẾ (PRODUCTION)")
print("="*50)

# ---------- CHẤM ĐIỂM XGBOOST ----------
pred_xgb_scaled = model_xgb.predict(X_test_scaled)
# XGBoost của bro đang trả ra mảng 3 số dự báo. Tui lấy cột 0 (T+1) để so với thực tế
pred_xgb_1h = scaler.inverse_transform(pred_xgb_scaled)[:, 0] 
pred_xgb_1h = np.maximum(0, pred_xgb_1h) # Khử số âm

r2_xgb = r2_score(y_test_real, pred_xgb_1h)
rmse_xgb = np.sqrt(mean_squared_error(y_test_real, pred_xgb_1h))
mae_xgb = mean_absolute_error(y_test_real, pred_xgb_1h)

print(f"🏆 MÔ HÌNH XGBOOST:")
print(f"   - R² Score : {r2_xgb:.4f}")
print(f"   - RMSE     : {rmse_xgb:.2f} µg/m³")
print(f"   - MAE      : {mae_xgb:.2f} µg/m³\n")

# ---------- CHẤM ĐIỂM RIDGE ----------
pred_ridge_scaled = model_ridge.predict(X_test_scaled)
pred_ridge_1h = scaler.inverse_transform(pred_ridge_scaled)[:, 0]
pred_ridge_1h = np.maximum(0, pred_ridge_1h)

r2_ridge = r2_score(y_test_real, pred_ridge_1h)
rmse_ridge = np.sqrt(mean_squared_error(y_test_real, pred_ridge_1h))
mae_ridge = mean_absolute_error(y_test_real, pred_ridge_1h)

print(f"🏆 MÔ HÌNH RIDGE:")
print(f"   - R² Score : {r2_ridge:.4f}")
print(f"   - RMSE     : {rmse_ridge:.2f} µg/m³")
print(f"   - MAE      : {mae_ridge:.2f} µg/m³")
print("="*50)

🔄 Đang nạp Model và Scaler...
❌ Lỗi nạp file: [Errno 2] No such file or directory: 'xgboost_model.pkl'


NameError: name 'scaler' is not defined